# NLP Lab Task 5: N-gram Language Models

**Corpus**

| Split | Sentences |
|---|---|
| Train | the cat sat on the mat · the cat ate the fish · the dog sat on the rug · the dog ate the bone |
| Test | the cat sat on the rug · the fish sat on the mat |

**Conventions used throughout**

- Everything is lowercased; `<s>` and `</s>` are added to every sentence.
- The vocabulary is $V$ = all words **plus** `</s>` (no `<s>`), because `<s>` is only ever a context, never something the model predicts.
- Natural log (`ln`), results rounded to 4 decimals, standard library only.
- Values smaller than 0.0001 would print as `0.0000`, so they are shown in scientific notation (e.g. `2.5499e-07`). Exact zeros are still shown as `0.0000`.

In [1]:
import math
import itertools
from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test  = ["the cat sat on the rug", "the fish sat on the mat"]

def fmt(x):
    """Round to 4 decimals; tiny values use scientific notation so they do not print as 0.0000."""
    if x == math.inf:  return "inf"
    if x == -math.inf: return "-inf"
    if x == 0:         return "0.0000"
    if abs(x) < 0.0001: return f"{x:.4e}"
    return f"{x:.4f}"

## Task 1: Tokenization

**Statement.** A language model needs a fixed, repeatable way of turning text into tokens. `tokenize` lowercases the sentence, splits on spaces, and wraps the result in `<s>` (start) and `</s>` (end). The start marker gives the first word a context, and the end marker lets the model learn *when a sentence stops*, so probabilities over whole sentences are well defined.

`build_vocab` collects every word plus `</s>`, and deliberately leaves out `<s>`.

In [2]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]

train_tok = [tokenize(s) for s in train]
test_tok  = [tokenize(s) for s in test]

print("Training sentences:")
for t in train_tok: print("  ", t)
print("Test sentences:")
for t in test_tok: print("  ", t)

n_with    = sum(len(t) for t in train_tok)
n_without = sum(len(t) - 2 for t in train_tok)
print("\nTokens in training corpus WITH markers   :", n_with)
print("Tokens in training corpus WITHOUT markers:", n_without)

def build_vocab(sentences):
    vocab = set()
    for s in sentences:
        for tok in tokenize(s):
            if tok != "<s>":          # V = all words plus </s>, no <s>
                vocab.add(tok)
    return vocab

vocab = build_vocab(train)
V = len(vocab)
print("\nV =", V)
print("Sorted vocabulary:", sorted(vocab))

Training sentences:
   ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
   ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
   ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
   ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
Test sentences:
   ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
   ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Tokens in training corpus WITH markers   : 30
Tokens in training corpus WITHOUT markers: 22

V = 11
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


**Observations**

- The four training sentences contain **30 tokens with markers** (6+2, 5+2, 6+2, 5+2 → 8+7+8+7) and **22 tokens without** them. The difference of 8 is the 4 `<s>` and 4 `</s>` markers.
- $|V| = 11$: ten words (`ate, bone, cat, dog, fish, mat, on, rug, sat, the`) plus `</s>`.
- Every test word appears in the training vocabulary, so there are no out-of-vocabulary words. Any trouble later comes from unseen *combinations*, not unseen words.

## Task 2: Generating N-grams

**Statement.** An N-gram is a window of $n$ consecutive tokens. `get_ngrams(tokens, n)` slides that window across the sentence and returns a list of tuples.

A sentence of $m$ tokens has exactly $m - n + 1$ windows, because the window can start at positions $0, 1, \dots, m-n$. If $n > m$ there is no valid window, so the function returns an empty list instead of raising an error.

In [3]:
def get_ngrams(tokens, n):
    if n < 1 or n > len(tokens):
        return []                       # too short: no N-grams, no crash
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

first = train_tok[0]
print("Sentence:", first)
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(f"{name}: {get_ngrams(first, n)}")

print("\nVerification: number of N-grams == m - n + 1")
print(f"{'sentence':<32}{'m':>3}{'n=1':>6}{'n=2':>6}{'n=3':>6}")
for tokens in train_tok + test_tok:
    m = len(tokens)
    row = []
    for n in (1, 2, 3):
        got = len(get_ngrams(tokens, n))
        assert got == m - n + 1
        row.append(got)
    print(f"{' '.join(tokens[1:-1]):<32}{m:>3}{row[0]:>6}{row[1]:>6}{row[2]:>6}")
print("All assertions passed.")

print("\nn larger than sentence length:")
print("get_ngrams(first, 50)       ->", get_ngrams(first, 50))
print("get_ngrams(['<s>','</s>'],3) ->", get_ngrams(["<s>", "</s>"], 3))
print("get_ngrams([], 1)           ->", get_ngrams([], 1))

Sentence: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

Verification: number of N-grams == m - n + 1
sentence                          m   n=1   n=2   n=3
the cat sat on the mat            8     8     7     6
the cat ate the fish              7     7     6     5
the dog sat on the rug            8     8     7     6
the dog ate the bone              7     7     6     5
the cat sat on the rug            8     8     7     6
the fish sat on the mat           8     8     7     6
All assertions passed.

n larger than sentence length:
get_ngrams(first, 50)       -> []
get_ngrams(['<s>','</s>'],3) -> []
get_ngrams([], 1)         

**Observations**

- For the first sentence ($m = 8$) we get 8 unigrams, 7 bigrams and 6 trigrams, exactly $m-n+1$.
- The assertion holds for all 6 sentences and all three values of $n$.
- Asking for 50-grams (or trigrams of a 2-token sentence) returns `[]`. The slice range is simply empty, and the explicit guard also protects against `n < 1`.

## Task 3: Counting N-grams

**Statement.** Maximum-likelihood estimates are just ratios of counts, so the model is fully described by three `Counter` tables built from the training corpus. Ties in the sorted tables are broken alphabetically so the output is deterministic.

`count_of` uses `Counter.get(key, 0)`, so an unseen N-gram returns `0` instead of raising a `KeyError`. (Indexing a `Counter` directly would also return 0, but `.get` makes the intent explicit and does not insert new keys.)

In [4]:
uni = Counter(g for t in train_tok for g in get_ngrams(t, 1))
bi  = Counter(g for t in train_tok for g in get_ngrams(t, 2))
tri = Counter(g for t in train_tok for g in get_ngrams(t, 3))

def show_table(title, counter):
    print(f"--- {title} (highest to lowest) ---")
    for gram, c in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"{' '.join(gram):<22}{c:>3}")
    print()

show_table("Unigram counts", uni)
show_table("Bigram counts", bi)
show_table("Trigram counts", tri)

top5 = sorted(bi.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
print("5 most common bigrams:")
for gram, c in top5: print(f"  {gram}  {c}")

print("\nDistinct unigrams:", len(uni))
print("Distinct bigrams :", len(bi))
print("Distinct trigrams:", len(tri))

def count_of(ngram):
    ngram = (ngram,) if isinstance(ngram, str) else tuple(ngram)
    table = {1: uni, 2: bi, 3: tri}.get(len(ngram), {})
    return table.get(ngram, 0)

print("\ncount_of(('the','cat'))  =", count_of(("the", "cat")), " (seen)")
print("count_of(('fish','sat')) =", count_of(("fish", "sat")), " (unseen)")

--- Unigram counts (highest to lowest) ---
the                     8
</s>                    4
<s>                     4
ate                     2
cat                     2
dog                     2
on                      2
sat                     2
bone                    1
fish                    1
mat                     1
rug                     1

--- Bigram counts (highest to lowest) ---
<s> the                 4
ate the                 2
on the                  2
sat on                  2
the cat                 2
the dog                 2
bone </s>               1
cat ate                 1
cat sat                 1
dog ate                 1
dog sat                 1
fish </s>               1
mat </s>                1
rug </s>                1
the bone                1
the fish                1
the mat                 1
the rug                 1

--- Trigram counts (highest to lowest) ---
<s> the cat             2
<s> the dog             2
sat on the              2
ate the bone

**Observations**

- `the` is by far the most frequent word (8 of 30 tokens). `<s>` and `</s>` each occur 4 times, once per sentence.
- The most common bigram is `<s> the` (4): every sentence starts with "the".
- Distinct counts: **12 unigrams** (11 in $V$ plus `<s>`), **18 bigrams**, **19 trigrams**. The number of distinct N-grams grows with $n$, while the counts per N-gram shrink. That is the sparsity problem in miniature.
- `count_of(('the','cat')) = 2` (seen), `count_of(('fish','sat')) = 0` (unseen).

## Task 4: Unigram model

**Statement.** The unigram model ignores context:

$$P(w) = \frac{C(w)}{N}, \qquad P(w_1 \dots w_n) = \prod_{i} P(w_i)$$

Here $N$ is the number of tokens the model can *predict*: the 22 words plus the 4 `</s>` markers, so **$N = 26$**. `<s>` is excluded because it is never predicted. This choice is what makes the probabilities over $V$ sum to exactly 1 (using 30 would leave 4/30 of the mass on `<s>`).

A sentence is scored over its words and the final `</s>`.

In [5]:
# Unigram model: <s> is never predicted, so it is excluded from N.
N = sum(c for (w,), c in uni.items() if w != "<s>")
print("N (word tokens + </s>) =", N)

def unigram_prob(w):
    return uni.get((w,), 0) / N if w != "<s>" else 0.0

print(f"\n{'word':<8}{'count':>6}{'P(w)':>10}")
for w in sorted(vocab, key=lambda w: (-uni[(w,)], w)):
    print(f"{w:<8}{uni[(w,)]:>6}{fmt(unigram_prob(w)):>10}")
total = sum(unigram_prob(w) for w in vocab)
assert math.isclose(total, 1.0)
print("Sum of probabilities =", fmt(total), "-> assertion passed")

def unigram_sentence_prob(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:     # skip <s>, include </s>
        p *= unigram_prob(w)
    return p

print()
for s in test:
    print(f"P_uni('{s}') = {fmt(unigram_sentence_prob(s))}")

# Word order does not matter for a unigram model
words = test[0].split(" ")
orders = list(itertools.permutations(words))
base = unigram_sentence_prob(test[0])
probs = [unigram_sentence_prob(" ".join(o)) for o in orders]
assert all(math.isclose(p, base) for p in probs)
print(f"\nPermutations of '{test[0]}': {len(orders)} total, {len(set(orders))} distinct word orders")
print("All of them have the same unigram probability:", fmt(base))
for o in list(dict.fromkeys(orders))[:4]:
    print(f"  {' '.join(o):<26}{fmt(unigram_sentence_prob(' '.join(o)))}")

N (word tokens + </s>) = 26

word     count      P(w)
the          8    0.3077
</s>         4    0.1538
ate          2    0.0769
cat          2    0.0769
dog          2    0.0769
on           2    0.0769
sat          2    0.0769
bone         1    0.0385
fish         1    0.0385
mat          1    0.0385
rug          1    0.0385
Sum of probabilities = 1.0000 -> assertion passed

P_uni('the cat sat on the rug') = 2.5499e-07
P_uni('the fish sat on the mat') = 1.2749e-07

Permutations of 'the cat sat on the rug': 720 total, 360 distinct word orders
All of them have the same unigram probability: 2.5499e-07
  the cat sat on the rug    2.5499e-07
  the cat sat on rug the    2.5499e-07
  the cat sat the on rug    2.5499e-07
  the cat sat the rug on    2.5499e-07


**Observations**

- $P(\text{the}) = 8/26 = 0.3077$, the largest by far. The probabilities sum to exactly 1.
- Both test sentences get a **very small** probability (about $2.5\times10^{-7}$ and $1.3\times10^{-7}$). That is normal for a product of 7 probabilities and is why we move to log space later.
- The second sentence is exactly **half** as likely as the first. The two sentences share `the, sat, on, the, </s>` and differ only in `cat`(2) vs `fish`(1) and `rug`(1) vs `mat`(1). The `cat`→`fish` swap halves the probability and the `rug`→`mat` swap changes nothing.
- **Word order is invisible to a unigram model.** The 6 words can be arranged in 720 ways (360 distinct strings because `the` appears twice), and every one has probability $2.5499\times10^{-7}$. "the rug on sat cat the" scores the same as the grammatical sentence, so this model cannot capture syntax at all.

## Task 5: Bigram model (MLE)

**Statement.** The bigram model conditions each word on the one before it:

$$P(w \mid v) = \frac{C(v, w)}{C(v)}$$

By the chain rule with a first-order Markov assumption,

$$P(w_1 \dots w_n) \approx \prod_{i=1}^{n+1} P(w_i \mid w_{i-1}), \qquad w_0 = \texttt{<s>},\; w_{n+1} = \texttt{</s>}$$

`next_word_distribution(prev)` returns every observed continuation with its probability, and `predict_next` returns the arg-max with ties broken alphabetically.

In [6]:
def bigram_prob(prev, w):
    c = uni.get((prev,), 0)
    return bi.get((prev, w), 0) / c if c else 0.0

def next_word_distribution(prev):
    c = uni.get((prev,), 0)
    return {w: n / c for (p, w), n in sorted(bi.items()) if p == prev}

dist = next_word_distribution("the")
print("P(next | 'the'):")
for w, p in sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {w:<6}{fmt(p)}")
assert math.isclose(sum(dist.values()), 1.0)
print("Sum =", fmt(sum(dist.values())), "-> assertion passed")

def bigram_sentence_prob(sentence):
    p = 1.0
    for prev, w in get_ngrams(tokenize(sentence), 2):
        p *= bigram_prob(prev, w)
    return p

print()
for s in test:
    print(f"P_bigram('{s}') = {fmt(bigram_sentence_prob(s))}")

def predict_next(prev):
    d = next_word_distribution(prev)
    if not d:
        return None                                   # no continuation seen
    return min(d, key=lambda w: (-d[w], w))           # highest prob, ties alphabetical

print("\nNext-word predictions:")
for w in ["<s>", "the", "cat", "dog", "sat", "on", "ate", "fish", "</s>"]:
    print(f"  {w:<6}-> {predict_next(w)}")

P(next | 'the'):
  cat   0.2500
  dog   0.2500
  bone  0.1250
  fish  0.1250
  mat   0.1250
  rug   0.1250
Sum = 1.0000 -> assertion passed

P_bigram('the cat sat on the rug') = 0.0156
P_bigram('the fish sat on the mat') = 0.0000

Next-word predictions:
  <s>   -> the
  the   -> cat
  cat   -> ate
  dog   -> ate
  sat   -> on
  on    -> the
  ate   -> the
  fish  -> </s>
  </s>  -> None


**Observations**

- After `the`, the model splits probability as `cat` 0.25, `dog` 0.25 and four words at 0.125 each. Eight tokens of `the` are followed by six different words, and the distribution sums to 1.
- **Test 1** ("the cat sat on the rug") has probability $1\cdot\tfrac14\cdot\tfrac12\cdot1\cdot1\cdot\tfrac18\cdot1 = 0.0156$. Every factor is a bigram seen in training.
- **Test 2** ("the fish sat on the mat") has probability **0**, because the bigram `fish sat` never occurs, so $P(\text{sat}\mid\text{fish}) = 0$ and one zero factor wipes out the whole product. This is the problem Tasks 7–9 address.
- Predictions: `the → cat` and `cat → ate` are both **ties** (0.25 each for cat/dog, and `ate`/`sat` at 0.5 each), resolved alphabetically. `</s>` returns `None` because nothing follows the end marker.

## Task 6: Trigram model

**Statement.** The trigram model uses two words of context:

$$P(w_3 \mid w_1, w_2) = \frac{C(w_1, w_2, w_3)}{C(w_1, w_2)}$$

There is no word before the first token, so the first word is predicted with the **bigram** model given `<s>`; every later word uses the trigram model. (The alternative is to pad with two `<s>` symbols, but the task specifies the bigram start.)

In [7]:
def trigram_prob(w1, w2, w3):
    c = bi.get((w1, w2), 0)
    return tri.get((w1, w2, w3), 0) / c if c else 0.0

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = bigram_prob(tokens[0], tokens[1])             # first word given <s>
    for w1, w2, w3 in get_ngrams(tokens, 3):          # every other word
        p *= trigram_prob(w1, w2, w3)
    return p

for s in test:
    print(f"P_trigram('{s}') = {fmt(trigram_sentence_prob(s))}")

print(f"\n{'N-gram type':<14}{'distinct':>9}")
for name, table in [("unigrams", uni), ("bigrams", bi), ("trigrams", tri)]:
    print(f"{name:<14}{len(table):>9}")

print()
tot_ub = tot_ut = 0
for s in test:
    toks = tokenize(s)
    ub = [g for g in get_ngrams(toks, 2) if count_of(g) == 0]
    ut = [g for g in get_ngrams(toks, 3) if count_of(g) == 0]
    tot_ub += len(ub); tot_ut += len(ut)
    print(f"'{s}'")
    print(f"  unseen bigrams  ({len(ub)} of {len(get_ngrams(toks, 2))}): {ub}")
    print(f"  unseen trigrams ({len(ut)} of {len(get_ngrams(toks, 3))}): {ut}")
    if len(ut) > len(ub):   verdict = "the TRIGRAM model has more unseen N-grams"
    elif len(ub) > len(ut): verdict = "the BIGRAM model has more unseen N-grams"
    else:                   verdict = "tie (equal number of unseen N-grams)"
    print(f"  -> {verdict}")
if tot_ut > tot_ub:   verdict = "the TRIGRAM model has more unseen N-grams"
elif tot_ub > tot_ut: verdict = "the BIGRAM model has more unseen N-grams"
else:                 verdict = "tie"
print(f"\nBoth test sentences together: {tot_ub} unseen bigrams vs {tot_ut} unseen trigrams -> {verdict}")

P_trigram('the cat sat on the rug') = 0.1250
P_trigram('the fish sat on the mat') = 0.0000

N-gram type    distinct
unigrams             12
bigrams              18
trigrams             19

'the cat sat on the rug'
  unseen bigrams  (0 of 7): []
  unseen trigrams (0 of 6): []
  -> tie (equal number of unseen N-grams)
'the fish sat on the mat'
  unseen bigrams  (1 of 7): [('fish', 'sat')]
  unseen trigrams (3 of 6): [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
  -> the TRIGRAM model has more unseen N-grams

Both test sentences together: 1 unseen bigrams vs 3 unseen trigrams -> the TRIGRAM model has more unseen N-grams


**Observations**

- Test 1 scores **0.1250** under the trigram model versus 0.0156 under the bigram model, so the trigram model is **more confident** on a sentence it has effectively seen. The extra context removes ambiguity: after `<s> the` only `cat`/`dog` occur (0.5 each, versus 0.25 under the bigram model, which spreads `the` over six successors), and after `on the` only `mat`/`rug` occur (0.5 each, versus 0.125 under the bigram model). The bigram model wastes probability on branches such as `the → bone` that the longer context rules out.
- Test 2 is again **0**.
- The trigram model has **more unseen N-grams** on test 2: 3 unseen trigrams (`<s> the fish`, `the fish sat`, `fish sat on`) against 1 unseen bigram (`fish sat`). One missing bigram is "counted" by several overlapping trigrams, so higher-order models fail *more often* on new text even though they are sharper on seen text. This is the bias–variance trade-off of N-gram order.

## Task 7: Zero-probability problem

**Statement.** MLE gives probability 0 to anything it has not seen. Because sentence probability is a product, a single unseen bigram makes the whole sentence impossible, even if every word is common and the sentence is perfectly sensible.

`unseen_bigrams` lists such bigrams. `safe_log` returns $-\infty$ for $p = 0$ (as `math.log(0)` raises `ValueError`), so log-probabilities of impossible sentences stay well defined.

In [8]:
def unseen_bigrams(sentence):
    return [g for g in get_ngrams(tokenize(sentence), 2) if count_of(g) == 0]

for s in test:
    p = bigram_sentence_prob(s)
    print(f"'{s}'")
    print("  unseen bigrams       :", unseen_bigrams(s))
    print("  bigram MLE probability:", fmt(p))
assert bigram_sentence_prob(test[1]) == 0
assert bigram_sentence_prob(test[0]) > 0
print("\nConfirmed: the second test sentence has probability exactly 0.")

words = sorted(w for w in vocab if w != "</s>")
possible = len(words) ** 2
seen = sum(1 for a in words for b in words if bi.get((a, b), 0) > 0)
never = possible - seen
print(f"\nPossible word pairs (markers excluded): {len(words)} x {len(words)} = {possible}")
print(f"Seen in training : {seen}")
print(f"Never seen       : {never}  ({fmt(100 * never / possible)}%)")

def safe_log(p):
    return -math.inf if p == 0 else math.log(p)

def sentence_logprob(sentence):
    return sum(safe_log(bigram_prob(a, b)) for a, b in get_ngrams(tokenize(sentence), 2))

print()
for s in test:
    print(f"ln P_bigram('{s}') = {fmt(sentence_logprob(s))}")

'the cat sat on the rug'
  unseen bigrams       : []
  bigram MLE probability: 0.0156
'the fish sat on the mat'
  unseen bigrams       : [('fish', 'sat')]
  bigram MLE probability: 0.0000

Confirmed: the second test sentence has probability exactly 0.

Possible word pairs (markers excluded): 10 x 10 = 100
Seen in training : 13
Never seen       : 87  (87.0000%)

ln P_bigram('the cat sat on the rug') = -4.1589
ln P_bigram('the fish sat on the mat') = -inf


**Observations**

- Test 1 has no unseen bigrams (probability 0.0156). Test 2 has one, `fish sat`, so its probability is **exactly 0** and its log-probability is $-\infty$.
- Of the $10 \times 10 = 100$ possible word pairs, only 13 appear in training, so **87.0 %** of all word pairs have a zero MLE probability. A 4-sentence corpus covers a tiny part of the space, and even a large corpus leaves many grammatical pairs unseen.
- "Never seen" does not mean "impossible". `fish sat` is a perfectly good English bigram. MLE confuses *absence of evidence* with *evidence of absence*, which is exactly what smoothing corrects.

## Task 8: Smoothing

**Statement.** Smoothing moves a little probability mass from seen events to unseen ones.

- **Add-one (Laplace):** $P(w \mid v) = \dfrac{C(v,w) + 1}{C(v) + V}$
- **Add-k:** $P(w \mid v) = \dfrac{C(v,w) + k}{C(v) + kV}$, where $k \le 1$ takes less mass away from seen bigrams.

The denominator grows by $V$ (or $kV$) because we add pseudo-counts to **all** $V$ possible next words, which keeps each distribution summing to 1. We check that for every legal context. `</s>` is not a context (nothing follows it), so the check covers `<s>` and the 10 words.

In [9]:
def bigram_prob_addone(prev, w):
    return (bi.get((prev, w), 0) + 1) / (uni.get((prev,), 0) + V)

print(f"{'quantity':<16}{'MLE':>10}{'add-one':>10}")
for prev, w in [("fish", "sat"), ("the", "cat")]:
    print(f"{'P(' + w + ' | ' + prev + ')':<16}{fmt(bigram_prob(prev, w)):>10}{fmt(bigram_prob_addone(prev, w)):>10}")

# </s> is not a context: nothing follows it, so only <s> and real words are checked
prev_words = ["<s>"] + sorted(w for w in vocab if w != "</s>")
for prev in prev_words:
    total = sum(bigram_prob_addone(prev, w) for w in vocab)
    assert math.isclose(total, 1.0), (prev, total)
print(f"\nAdd-one distributions sum to 1 for all {len(prev_words)} possible previous words.")

def bigram_prob_addk(prev, w, k):
    return (bi.get((prev, w), 0) + k) / (uni.get((prev,), 0) + k * V)

print(f"\n{'k':<8}{'P(sat | fish)':>14}")
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k:<8}{fmt(bigram_prob_addk('fish', 'sat', k)):>14}")

quantity               MLE   add-one
P(sat | fish)       0.0000    0.0833
P(cat | the)        0.2500    0.1579

Add-one distributions sum to 1 for all 11 possible previous words.

k        P(sat | fish)
1               0.0833
0.5             0.0769
0.1             0.0476
0.01            0.0090


**Observations**

- $P(\text{sat}\mid\text{fish})$ goes from **0 to 0.0833** under add-one ($1/(1+11)$), so the zero is repaired.
- The price is paid by seen events: $P(\text{cat}\mid\text{the})$ falls from **0.2500 to 0.1579** ($3/19$). With such a small corpus and $V = 11$, add-one moves a lot of mass to unseen pairs.
- Smaller $k$ gives the unseen event less mass: $P(\text{sat}\mid\text{fish})$ shrinks from 0.0833 ($k=1$) to 0.0090 ($k=0.01$), approaching the MLE value of 0 as $k \to 0$.
- All 11 smoothed distributions sum to 1, so the smoothed model is a valid probability model.

## Task 9: Perplexity and model comparison

**Statement.** Perplexity is the inverse probability of the test data, normalised by the number of predicted tokens $M$:

$$\mathrm{PP} = \exp\!\Big(-\frac{1}{M}\sum_{i=1}^{M}\ln P(w_i \mid \text{context})\Big)$$

Lower is better, and it can be read as the model's effective branching factor. Here $M = 14$ (7 predicted tokens in each of the 2 test sentences: 6 words + `</s>`). If any probability is 0 the perplexity is infinite.

The unigram add-one model uses $P(w) = \dfrac{C(w)+1}{N+V}$ with $N = 26$.

**Part 4 (validation).** Training on only the first three sentences, $k$ is tuned on the **fourth** sentence (validation), then that $k$ is used on the test set. This avoids choosing hyper-parameters on the very data we report, and we compare it with the "cheating" best $k$ chosen directly on test.

In [10]:
def perplexity(sentences, prob_fn):
    """prob_fn(prev, w) -> probability. Every predicted token counts (words and </s>)."""
    log_sum, n = 0.0, 0
    for s in sentences:
        for prev, w in get_ngrams(tokenize(s), 2):
            p = prob_fn(prev, w)
            if p == 0:
                return math.inf
            log_sum += math.log(p)
            n += 1
    return math.exp(-log_sum / n)

n_test = sum(len(tokenize(s)) - 1 for s in test)
print("Predicted tokens in the test set:", n_test)

def unigram_prob_addone(prev, w):
    return (uni.get((w,), 0) + 1) / (N + V)

results = [
    ("Bigram MLE",     perplexity(test, bigram_prob)),
    ("Bigram add-one", perplexity(test, bigram_prob_addone)),
    ("Unigram add-one", perplexity(test, unigram_prob_addone)),
]
print(f"\n{'model':<18}{'perplexity':>12}")
for name, pp in results:
    print(f"{name:<18}{fmt(pp):>12}")

ks = [1, 0.5, 0.1, 0.01]
pp_k = {k: perplexity(test, lambda a, b, k=k: bigram_prob_addk(a, b, k)) for k in ks}
print(f"\n{'k':<8}{'perplexity':>12}")
for k in ks:
    print(f"{k:<8}{fmt(pp_k[k]):>12}")
best_k = min(pp_k, key=pp_k.get)
print("Best k (full training data, on test):", best_k)

# ---- Validation experiment: train on first three sentences only ----
def counts_from(sentences):
    u, b = Counter(), Counter()
    for s in sentences:
        t = tokenize(s)
        u.update(get_ngrams(t, 1))
        b.update(get_ngrams(t, 2))
    return u, b

def make_addk(u, b, k):
    return lambda prev, w: (b.get((prev, w), 0) + k) / (u.get((prev,), 0) + k * V)

u3, b3 = counts_from(train[:3])         # V still comes from all four sentences
val = [train[3]]

val_pp  = {k: perplexity(val,  make_addk(u3, b3, k)) for k in ks}
test_pp = {k: perplexity(test, make_addk(u3, b3, k)) for k in ks}
k_val  = min(val_pp,  key=val_pp.get)
k_test = min(test_pp, key=test_pp.get)

print(f"\nTrained on 3 sentences, V = {V}")
print(f"{'k':<8}{'validation PP':>15}{'test PP':>12}")
for k in ks:
    print(f"{k:<8}{fmt(val_pp[k]):>15}{fmt(test_pp[k]):>12}")
print(f"\nBest k on validation sentence : {k_val}  -> test perplexity {fmt(test_pp[k_val])}")
print(f"Best k picked on the test set : {k_test}  -> test perplexity {fmt(test_pp[k_test])}")
print("Same k chosen:", k_val == k_test, "| gap in test perplexity:", fmt(test_pp[k_val] - test_pp[k_test]))

# Reference: the same two choices of k on the model trained on ALL four sentences
print(f"\nFor reference, full 4-sentence model: k={k_val} -> {fmt(pp_k[k_val])},  k={k_test} -> {fmt(pp_k[k_test])}")

Predicted tokens in the test set: 14

model               perplexity
Bigram MLE                 inf
Bigram add-one          5.7969
Unigram add-one         9.1370

k         perplexity
1             5.7969
0.5           4.5703
0.1           2.9559
0.01          2.6272
Best k (full training data, on test): 0.01

Trained on 3 sentences, V = 11
k         validation PP     test PP
1                8.5857      5.7406
0.5              8.0163      4.4539
0.1              8.0260      2.7778
0.01            13.1304      2.4258

Best k on validation sentence : 0.5  -> test perplexity 4.4539
Best k picked on the test set : 0.01  -> test perplexity 2.4258
Same k chosen: False | gap in test perplexity: 2.0280

For reference, full 4-sentence model: k=0.5 -> 4.5703,  k=0.01 -> 2.6272


**Observations**

- **Bigram MLE: ∞.** The single unseen bigram `fish sat` ruins the whole test set. **Bigram add-one: 5.7969** is finite and clearly better than **unigram add-one: 9.1370**, so context still helps after smoothing: the bigram model is about as unsure as choosing among ~6 words rather than ~9.
- **Add-k:** perplexity decreases as $k$ falls (5.7969 → 4.5703 → 2.9559 → 2.6272), so the **best $k$ among those tried is 0.01**. This is a property of the grid: checking a finer grid separately shows the true minimum is near $k \approx 0.02$ (about 2.60), and perplexity rises again for smaller $k$ and reaches ∞ at $k = 0$, which is the MLE with its zero.
- **Why small $k$ wins here:** the test set contains 13 seen bigrams and only 1 unseen one, so it rewards models that stay close to the counts.
- **Validation experiment.** Trained on three sentences, the validation sentence ("the dog ate the bone") contains three unseen bigrams (`dog ate`, `the bone`, `bone </s>`), so it prefers a *larger* $k$: the best validation $k$ is **0.5** (8.0163, with 0.1 almost tied at 8.0260) and gives a test perplexity of **4.4539**. Choosing $k$ directly on the test set picks **0.01** and gives **2.4258**.
- **Interpretation.** The test-chosen $k$ is better *on this test set* by construction, because it was tuned on it, and its number is an optimistic estimate that will not generalise. The validation-chosen figure (4.4539) is the honest one. The gap (about 2.03) here mainly reflects that a single validation sentence is a very small and unrepresentative sample (it has 3 unseen bigrams out of 6, while the test set has only 1 out of 14). With a realistic validation set the two choices would normally agree much more closely.

## Summary

| Model | Test perplexity |
|---|---|
| Bigram MLE | ∞ |
| Unigram add-one | 9.1370 |
| Bigram add-one ($k=1$) | 5.7969 |
| Bigram add-k, best $k=0.01$ (4-sentence training) | 2.6272 |
| Bigram add-k, $k=0.5$ picked on validation (3-sentence training) | 4.4539 |

**Takeaways:** (1) context makes a model sharper (bigram < unigram perplexity); (2) higher order means more unseen N-grams, hence more zeros; (3) MLE assigns zero to unseen events and gives infinite perplexity; (4) smoothing fixes this at the cost of some probability on seen events; (5) hyper-parameters such as $k$ must be tuned on held-out validation data, not on the test set.